# Contextual Retrieval Demo

**Contextual Retrieval** (Anthropic, September 2024) enriches each chunk with LLM-generated context before embedding.

**The Problem:** A chunk saying "The policy was updated in Q3" loses meaning without knowing which policy.

**The Solution:** Use an LLM to prepend context: "This chunk is from the Remote Work Policy document, Section 3..."

**Trade-off:** Higher indexing cost (1 LLM call per chunk) but better retrieval quality.

**Prerequisites:**
```bash
pip install langchain-text-splitters requests numpy
ollama pull qwen3:4b
ollama pull nomic-embed-text
```

In [1]:
# Setup
import subprocess
import requests
import numpy as np
import time

def check_ollama():
    try:
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            print("✓ Ollama running")
            if "qwen3" in result.stdout:
                print("✓ qwen3 available (for context generation)")
            if "nomic-embed-text" in result.stdout:
                print("✓ nomic-embed-text available (for embeddings)")
        return True
    except Exception as e:
        print(f"✗ Ollama not available: {e}")
        return False

check_ollama()

# Sample document with cross-references
SAMPLE_DOCUMENT = """
Acme Corporation Employee Handbook
Version 3.2 | Effective January 2024

Chapter 1: Remote Work Policy

1.1 Overview
This chapter establishes the company's guidelines for remote work arrangements. The policy applies to all full-time employees who have completed their probation period.

1.2 Eligibility
Employees must meet the following criteria:
- Completed 90 days of employment
- Performance rating of 3.0 or above
- Manager approval obtained in writing

1.3 Work Schedule
Remote employees must be available during core hours (10am-3pm local time). The flexibility provisions in Section 1.1 allow for adjusted schedules with manager approval.

1.4 Equipment
As referenced in Chapter 3, IT equipment is provided by the company. Remote workers receive the standard laptop package plus a $100 monthly stipend.

Chapter 2: Performance Management

2.1 Review Cycle
Performance reviews occur quarterly. Managers must complete evaluations within 2 weeks of the quarter end. The criteria from Chapter 1 regarding remote work eligibility are evaluated during these reviews.
"""

print(f"\nDocument: {len(SAMPLE_DOCUMENT)} characters")

✓ Ollama running
✓ qwen3 available (for context generation)
✓ nomic-embed-text available (for embeddings)

Document: 1069 characters


---

## 1. Traditional Chunking (Baseline)

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def get_embedding(text: str, model: str = "nomic-embed-text") -> list[float]:
    response = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": model, "prompt": text}
    )
    return response.json()["embedding"]

def cosine_similarity(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Traditional chunking
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)
chunks = splitter.split_text(SAMPLE_DOCUMENT)

print(f"Traditional Chunking: {len(chunks)} chunks")
print("=" * 60)

# Show chunks with cross-references
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(f"  \"{chunk[:80]}...\"")
    # Detect cross-references
    if any(ref in chunk.lower() for ref in ["section", "chapter", "referenced", "criteria from"]):
        print("  ⚠ Contains cross-reference")

# Embed traditionally
print("\nEmbedding chunks (traditional)...")
traditional_embeddings = [get_embedding(chunk) for chunk in chunks]
print(f"✓ {len(traditional_embeddings)} embeddings")

Traditional Chunking: 5 chunks

Chunk 1 (285 chars):
  "Acme Corporation Employee Handbook
Version 3.2 | Effective January 2024

Chapter..."
  ⚠ Contains cross-reference

Chunk 2 (169 chars):
  "1.2 Eligibility
Employees must meet the following criteria:
- Completed 90 days ..."

Chunk 3 (187 chars):
  "1.3 Work Schedule
Remote employees must be available during core hours (10am-3pm..."
  ⚠ Contains cross-reference

Chunk 4 (197 chars):
  "1.4 Equipment
As referenced in Chapter 3, IT equipment is provided by the compan..."
  ⚠ Contains cross-reference

Chunk 5 (221 chars):
  "2.1 Review Cycle
Performance reviews occur quarterly. Managers must complete eva..."
  ⚠ Contains cross-reference

Embedding chunks (traditional)...
✓ 5 embeddings


---

## 2. Contextual Retrieval (LLM-Enriched Chunks)

For each chunk, use an LLM to generate situating context.

In [3]:
def generate_context(document: str, chunk: str, model: str = "qwen3:4b") -> str:
    """
    Generate situating context for a chunk using LLM.
    
    This is the core of Contextual Retrieval:
    - LLM sees the full document + the specific chunk
    - Generates a brief context explaining what this chunk is about
    - Context is prepended to chunk before embedding
    """
    prompt = f"""<document>
{document[:2000]}  
</document>

Here is a chunk from this document:
<chunk>
{chunk}
</chunk>

Write a short context (1-2 sentences) that situates this chunk within the document.
Include: document title, section name, and what this chunk discusses.
No thinking, just the context. Be concise."""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    
    return response.json().get("response", "").strip()

def contextual_embed(document: str, chunks: list[str]) -> tuple[list[str], list[list[float]]]:
    """
    Apply contextual retrieval: generate context + embed.
    Returns contextualized chunks and their embeddings.
    """
    contextualized = []
    embeddings = []
    
    for i, chunk in enumerate(chunks):
        print(f"  Processing chunk {i+1}/{len(chunks)}...", end=" ")
        
        # Generate context
        start = time.time()
        context = generate_context(document, chunk)
        gen_time = time.time() - start
        
        # Combine context + chunk
        enriched = f"{context}\n\n{chunk}"
        contextualized.append(enriched)
        
        # Embed the enriched chunk
        embedding = get_embedding(enriched)
        embeddings.append(embedding)
        
        print(f"({gen_time:.1f}s)")
    
    return contextualized, embeddings

print("Contextual Retrieval: Enriching chunks with LLM-generated context")
print("=" * 60)
print("(This takes longer due to LLM calls per chunk)\n")

try:
    contextual_chunks, contextual_embeddings = contextual_embed(SAMPLE_DOCUMENT, chunks)
    print(f"\n✓ Generated {len(contextual_embeddings)} contextualized embeddings")
except Exception as e:
    print(f"Error: {e}")

Contextual Retrieval: Enriching chunks with LLM-generated context
(This takes longer due to LLM calls per chunk)

  Processing chunk 1/5... (46.1s)
  Processing chunk 2/5... (10.1s)
  Processing chunk 3/5... (51.7s)
  Processing chunk 4/5... (64.2s)
  Processing chunk 5/5... (40.7s)

✓ Generated 5 contextualized embeddings


---

## 3. Compare Contextualized vs Original Chunks

In [4]:
print("Before vs After Contextualization")
print("=" * 60)

for i in range(min(3, len(chunks))):
    print(f"\n--- Chunk {i+1} ---")
    print(f"\nOriginal:")
    print(f"  {chunks[i][:150]}...")
    print(f"\nContextualized:")
    print(f"  {contextual_chunks[i][:200]}...")

Before vs After Contextualization

--- Chunk 1 ---

Original:
  Acme Corporation Employee Handbook
Version 3.2 | Effective January 2024

Chapter 1: Remote Work Policy

1.1 Overview
This chapter establishes the comp...

Contextualized:
  The Acme Corporation Employee Handbook (Version 3.2, effective January 2024) section 1.1 Overview establishes the scope of remote work policies, stating they apply to full-time employees who have comp...

--- Chunk 2 ---

Original:
  1.2 Eligibility
Employees must meet the following criteria:
- Completed 90 days of employment
- Performance rating of 3.0 or above
- Manager approval ...

Contextualized:
  This section of the Acme Corporation Employee Handbook (Version 3.2) details the eligibility criteria for remote work under Chapter 1, specifying that employees must complete 90 days of employment, ac...

--- Chunk 3 ---

Original:
  1.3 Work Schedule
Remote employees must be available during core hours (10am-3pm local time). The flexibility provisions i

---

## 4. Retrieval Quality Comparison

In [5]:
def search(query: str, chunks: list[str], embeddings: list[list[float]], top_k: int = 2):
    """Search using cosine similarity."""
    query_emb = get_embedding(query)
    similarities = [cosine_similarity(query_emb, emb) for emb in embeddings]
    ranked = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)
    return [(chunks[i], score) for i, score in ranked[:top_k]]

# Test queries that benefit from context
test_queries = [
    "What are the remote work eligibility requirements at Acme?",
    "How does performance review relate to remote work?",
    "What equipment do remote workers get?",
]

print("Retrieval Comparison")
print("=" * 60)

for query in test_queries:
    print(f"\nQuery: \"{query}\"")
    print("-" * 50)
    
    # Traditional
    trad_results = search(query, chunks, traditional_embeddings, top_k=1)
    print(f"\nTraditional (score: {trad_results[0][1]:.3f}):")
    print(f"  \"{trad_results[0][0][:100]}...\"")
    
    # Contextual
    ctx_results = search(query, contextual_chunks, contextual_embeddings, top_k=1)
    print(f"\nContextual (score: {ctx_results[0][1]:.3f}):")
    # Show just the context part
    print(f"  \"{ctx_results[0][0][:150]}...\"")

Retrieval Comparison

Query: "What are the remote work eligibility requirements at Acme?"
--------------------------------------------------

Traditional (score: 0.735):
  "Acme Corporation Employee Handbook
Version 3.2 | Effective January 2024

Chapter 1: Remote Work Poli..."

Contextual (score: 0.839):
  "This section of the Acme Corporation Employee Handbook (Version 3.2) details the eligibility criteria for remote work under Chapter 1, specifying that..."

Query: "How does performance review relate to remote work?"
--------------------------------------------------

Traditional (score: 0.752):
  "2.1 Review Cycle
Performance reviews occur quarterly. Managers must complete evaluations within 2 we..."

Contextual (score: 0.784):
  "The Acme Corporation Employee Handbook (Version 3.2) section 2.1 "Review Cycle" details the quarterly performance review process, requiring managers t..."

Query: "What equipment do remote workers get?"
--------------------------------------------------

T

---

## 5. Cost Analysis

In [6]:
print("Cost Comparison (API pricing, per 1M tokens)")
print("=" * 60)
print("""
Method                  Embedding    LLM (indexing)   Total
------------------------------------------------------------
Traditional             $0.05        $0               $0.05
Late Chunking           $0.05        $0               $0.05
Contextual Retrieval    $0.05        ~$1.00           ~$1.05

With Ollama (local):
------------------------------------------------------------
All methods             Free         Free             Free

Trade-off Summary:
- Traditional: Fast, cheap, loses context
- Late Chunking: Fast, cheap, preserves context (~80% quality gain)
- Contextual: Slow indexing, expensive, maximum quality (~95% quality)

Recommendation:
- High-volume, cost-sensitive → Late Chunking
- Low-volume, quality-critical → Contextual Retrieval
- With Ollama → Contextual Retrieval (no cost penalty)
""")

Cost Comparison (API pricing, per 1M tokens)

Method                  Embedding    LLM (indexing)   Total
------------------------------------------------------------
Traditional             $0.05        $0               $0.05
Late Chunking           $0.05        $0               $0.05
Contextual Retrieval    $0.05        ~$1.00           ~$1.05

With Ollama (local):
------------------------------------------------------------
All methods             Free         Free             Free

Trade-off Summary:
- Traditional: Fast, cheap, loses context
- Late Chunking: Fast, cheap, preserves context (~80% quality gain)
- Contextual: Slow indexing, expensive, maximum quality (~95% quality)

Recommendation:
- High-volume, cost-sensitive → Late Chunking
- Low-volume, quality-critical → Contextual Retrieval
- With Ollama → Contextual Retrieval (no cost penalty)



---

## Summary

**Contextual Retrieval Process:**
1. Split document into chunks (any strategy)
2. For each chunk: LLM generates situating context
3. Prepend context to chunk
4. Embed the enriched chunk
5. At query time: search as normal

**When to use:**
- Documents with heavy cross-references
- Quality matters more than indexing speed
- Using local LLM (Ollama) eliminates cost concern
- Low-volume, high-value document sets

**Anthropic benchmarks:** -67% retrieval failures when combined with hybrid search and reranking.